# 项目：整理Netflix电影演员评分数据

## 分析目标

此数据分析的目的是，整理不同流派影视作品，比如喜剧片、动作片、科幻片中，各演员出演作品的平均IMDB评分，从而挖掘出各个流派中的高评分作品演员。

本实战项目的目的在于练习整理数据，从而得到可供下一步分析的数据。

## 简介

原始数据集记录了截止至2022年7月美国地区可观看的所有Netflix电视剧及电影数据。数据集包含两个数据表：`titles.csv`和`credits.csv`。

`titles.csv`包含电影及电视剧相关信息，包括影视作品ID、标题、类型、描述、流派、IMDB（一个国外的在线评分网站）评分，等等。`credits.csv`包含超过7万名出现在Netflix影视作品的导演及演员信息，包括名字、影视作品ID、人物名、演职员类型（导演/演员）等。

`titles.csv`每列的含义如下：
- id：影视作品ID。
- title：影视作品标题。
- show_type：作品类型，电视节目或电影。
- description：简短描述。
- release_year：发布年份。
- age_certification：适龄认证。
- runtime：每集电视剧或电影的长度。
- genres：流派类型列表。
- production_countries：出品国家列表。
- seasons：如果是电视剧，则是季数。
- imdb_id：IMDB的ID。
- imdb_score：IMDB的评分。
- imdb_votes：IMDB的投票数。
- tmdb_popularity：TMDB的流行度。
- tmdb_score：TMDB的评分。

`credits.csv`每列的含义如下：
- person_ID：演职员ID。
- id：参与的影视作品ID。
- name：姓名。
- character_name：角色姓名。
- role：演职员类型，演员或导演。

In [1]:
import pandas as pd

## 一、数据评估和清理

##### 在这一部分中，我们将对在上一部分建立的original_titles及original_creditsDataFrame所包含的数据进行评估和清理。  

主要从两个方面进行：结构和内容，即整齐度和干净度。  

数据的结构性问题指不符合“每个变量为一列，每个观察值为一行，每种类型的观察单位为一个表格”这三个标准；数据的内容性问题包括存在丢失数据、重复数据、无效数据等。

In [2]:
o_titles=pd.read_csv(r"C:\Users\30338\Desktop\数据处理学习\数据整理\titles.csv")
o_credits=pd.read_csv(r"C:\Users\30338\Desktop\数据处理学习\数据整理\credits.csv")

##### 为区分经过清理的数据和原始的数据，需创建新的变量`c_titles`，等于`o_titles`复制出的副本；创建新变量`c_credits`，等于`o_credits`复制出的副本。之后的清理步骤都将被运用在`c_titles`和`c_credits`上。

In [3]:
c_titles=o_titles.copy()
c_credits=o_credits.copy()

### 1.评估数据整齐度（结构）

In [4]:
c_titles.head(10)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600
5,ts22164,Monty Python's Flying Circus,SHOW,A British sketch comedy series with the shows ...,1969,TV-14,30,"['comedy', 'european']",['GB'],4.0,tt0063929,8.8,73424.0,17.617,8.306
6,tm70993,Life of Brian,MOVIE,"Brian Cohen is an average young Jewish man, bu...",1979,R,94,['comedy'],['GB'],NaN,tt0079470,8.0,395024.0,17.770,7.800
7,tm14873,Dirty Harry,MOVIE,When a madman dubbed 'Scorpio' terrorizes San ...,1971,R,102,"['thriller', 'action', 'crime']",['US'],NaN,tt0066999,7.7,155051.0,12.817,7.500
8,tm119281,Bonnie and Clyde,MOVIE,"In the 1930s, bored waitress Bonnie Parker fal...",1967,R,110,"['crime', 'drama', 'action']",['US'],NaN,tt0061418,7.7,112048.0,15.687,7.500
9,tm98978,The Blue Lagoon,MOVIE,Two small children and a ship's cook survive a...,1980,R,104,"['romance', 'action', 'drama']",['US'],NaN,tt0080453,5.8,69844.0,50.324,6.156


##### 通过对观察值进行浏览发现：genres和production两列的变量需要进行拆分

In [5]:
c_titles["genres"][1]

"['drama', 'crime']"

##### genres中的变量看似列表，提取出来后发现不是列表，而是字符串，因无法直接用value_counts统计各个值出现的次数。  
我们可以使用Python内置的eval函数，它可以把字符串转换成表达式，所以可以帮我们把表示列表的字符串转换成列表本身。
apply方法后面需要加一个函数，lambda在这里就是定义这个函数的，括号中的s表示该列的变量，由apply来应用eval函数将这一列的变量s改成表达式。

In [6]:
c_titles["genres"]=c_titles["genres"].apply(lambda s:eval(s))
c_titles.head(10)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,[documentation],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"[drama, crime]",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"[drama, action, thriller, european]",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"[fantasy, action, comedy]",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"[war, action]","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600
5,ts22164,Monty Python's Flying Circus,SHOW,A British sketch comedy series with the shows ...,1969,TV-14,30,"[comedy, european]",['GB'],4.0,tt0063929,8.8,73424.0,17.617,8.306
6,tm70993,Life of Brian,MOVIE,"Brian Cohen is an average young Jewish man, bu...",1979,R,94,[comedy],['GB'],NaN,tt0079470,8.0,395024.0,17.770,7.800
7,tm14873,Dirty Harry,MOVIE,When a madman dubbed 'Scorpio' terrorizes San ...,1971,R,102,"[thriller, action, crime]",['US'],NaN,tt0066999,7.7,155051.0,12.817,7.500
8,tm119281,Bonnie and Clyde,MOVIE,"In the 1930s, bored waitress Bonnie Parker fal...",1967,R,110,"[crime, drama, action]",['US'],NaN,tt0061418,7.7,112048.0,15.687,7.500
9,tm98978,The Blue Lagoon,MOVIE,Two small children and a ship's cook survive a...,1980,R,104,"[romance, action, drama]",['US'],NaN,tt0080453,5.8,69844.0,50.324,6.156


##### 转换成列表后就要用explode来拆分变量。str.split是针对字符串的，会报错  
explode方法针对整个表格，因此赋值等式左边就不需要引入具体列名。

In [7]:
c_titles=c_titles.explode("genres")
c_titles.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,documentation,['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,crime,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,drama,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,action,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300


##### 现在确定拆分没有问题的情况下，可以接着对production_countries也进行拆分

In [8]:
c_titles=c_titles.explode("production_countries")
c_titles.head(10)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,documentation,['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,crime,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,drama,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,action,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,thriller,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,european,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,fantasy,['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,action,['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,comedy,['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811


##### 现在c_titles的结构性问题已经完成解决，接下来解决c_credits的结构性问题

In [9]:
c_credits.head(10)

,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR
5,32267,tm84618,Peter Boyle,Wizard,ACTOR
6,519612,tm84618,Leonard Harris,Senator Charles Palantine,ACTOR
7,29068,tm84618,Diahnne Abbott,Concession Girl,ACTOR
8,519613,tm84618,Gino Ardito,Policeman at Rally,ACTOR
9,3308,tm84618,Martin Scorsese,Passenger Watching Silhouette,ACTOR


##### 从观察来看，c_credits应该不存在结构性问题，因此不再做结构性调整

### 2.数据干净度（内容）

#### 2.1  丢失数据
先用info对数据进行简单了解

In [10]:
c_titles.info()

<class 'pandas.DataFrame'>
Index: 15147 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    15147 non-null  str    
 1   title                 15146 non-null  str    
 2   type                  15147 non-null  str    
 3   description           15125 non-null  str    
 4   release_year          15147 non-null  int64  
 5   age_certification     9298 non-null   str    
 6   runtime               15147 non-null  int64  
 7   genres                15088 non-null  str    
 8   production_countries  15147 non-null  str    
 9   seasons               5923 non-null   float64
 10  imdb_id               14525 non-null  str    
 11  imdb_score            14399 non-null  float64
 12  imdb_votes            14375 non-null  float64
 13  tmdb_popularity       14995 non-null  float64
 14  tmdb_score            14614 non-null  float64
dtypes: float64(5), int64(2), str(8)
memo

##### 经过观察发现title、description、age_certification、genres、seasons、imdb_id、imdb_score、imdb_votes、tmdb_popularity、tmdb_score；这些列均出现了变量缺失，此后将对这些变量进行评估和清洗。   
同时release_year应该是日期类型而非数字类型，将在此后进行数据类型替换。在列后添加format=”%Y”使pandas将日期当成四位数的年份去解读，而不是时间戳。

In [11]:
c_titles["release_year"]=pd.to_datetime(c_titles["release_year"],format="%Y")
c_titles.info()

<class 'pandas.DataFrame'>
Index: 15147 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id                    15147 non-null  str           
 1   title                 15146 non-null  str           
 2   type                  15147 non-null  str           
 3   description           15125 non-null  str           
 4   release_year          15147 non-null  datetime64[us]
 5   age_certification     9298 non-null   str           
 6   runtime               15147 non-null  int64         
 7   genres                15088 non-null  str           
 8   production_countries  15147 non-null  str           
 9   seasons               5923 non-null   float64       
 10  imdb_id               14525 non-null  str           
 11  imdb_score            14399 non-null  float64       
 12  imdb_votes            14375 non-null  float64       
 13  tmdb_popularity       14995 non-n

##### 根据信息可以发现title、description、age_certification、genres、seasons、imdb_id、imdb_score、imdb_votes、tmdb_popularity、tmdb_score；这些列均出现了变量缺失；即对应标题、简短描述、适龄认证、作品派系、季数、IMDB的ID、IMDB分数、IMDB票数、TMDB的流行度、TMDB的评分；

In [12]:
c_credits.info()

<class 'pandas.DataFrame'>
RangeIndex: 77801 entries, 0 to 77800
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   person_id  77801 non-null  int64
 1   id         77801 non-null  str  
 2   name       77801 non-null  str  
 3   character  68029 non-null  str  
 4   role       77801 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.0 MB


##### 从上可知，c_credits共有五列 77801条数据  character列存在空值需要后续的评估清理  
person_id的数据类型不可以是int64应该为str，需要进行数据类型转换

In [13]:
c_credits["person_id"]=c_credits["person_id"].astype(str)
c_credits.info()

<class 'pandas.DataFrame'>
RangeIndex: 77801 entries, 0 to 77800
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   person_id  77801 non-null  str  
 1   id         77801 non-null  str  
 2   name       77801 non-null  str  
 3   character  68029 non-null  str  
 4   role       77801 non-null  str  
dtypes: str(5)
memory usage: 3.0 MB


##### 2.1.2 处理缺失数据  
title、description、age_certification、genres、seasons、imdb_id、imdb_score、imdb_votes、tmdb_popularity、tmdb_score；这些列均出现了变量缺失；  
即对应标题、简短描述、适龄认证、作品派系、季数、IMDB的ID、IMDB分数、IMDB票数、TMDB的流行度、TMDB的评分；  
根据电影信息的分析，发现标题、简短描述、适龄认证、季数、IMDB的ID、TMDB的流行度、TMDB的评分均不影响对高评分演员的挖掘。因此可以保留这些数据的空缺不予处理。  
但是genres和imdb_score、即作品派系和IMDB分数均会影响分析  
先输出imdb_score对缺失值进行查看

In [14]:
c_titles.query("imdb_score.isnull()")

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945-01-01,TV-MA,51,documentation,['US'],1.0,NaN,NaN,NaN,0.600,NaN
75,tm132164,Bill Hicks: Sane Man,MOVIE,Sane Man was filmed before Bill recorded ‘Dang...,1989-01-01,R,80,comedy,['US'],NaN,NaN,NaN,NaN,3.377,7.5
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,documentation,['JP'],12.0,NaN,NaN,NaN,7.730,7.8
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,family,['JP'],12.0,NaN,NaN,NaN,7.730,7.8
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,reality,['JP'],12.0,NaN,NaN,NaN,7.730,7.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5810,tm1225897,Social Man,MOVIE,Two competitive social media Influencers go he...,2021-01-01,NaN,96,drama,[],NaN,tt20198164,NaN,NaN,NaN,NaN
5833,ts307884,HQ Barbers,SHOW,When a family run barber shop in the heart of ...,2021-01-01,TV-14,24,comedy,['NG'],1.0,NaN,NaN,NaN,0.840,NaN
5840,tm1216735,Sun of the Soil,MOVIE,"In 14th-century Mali, an ambitious young royal...",2022-01-01,NaN,26,NaN,[],NaN,NaN,NaN,NaN,1.179,7.0
5844,tm1074617,Bling Empire - The Afterparty,MOVIE,"The stars of ""Bling Empire"" discuss the show's...",2021-01-01,NaN,35,NaN,['US'],NaN,NaN,NaN,NaN,NaN,NaN


##### 由于缺失值是关键信息，会对此后分析产生影响，因此应当将其删除并再次查看缺失总数

In [15]:
c_titles=c_titles.dropna(subset=["imdb_score"])
c_titles["imdb_score"].isnull().sum()

np.int64(0)

##### 然后提取出genres的变量缺失值进行查看并重复以上操作

In [16]:
c_titles.query("genres.isnull()")

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1813,ts77824,My Next Guest Needs No Introduction With David...,SHOW,TV legend David Letterman teams up with fascin...,2018-01-01,TV-MA,50,NaN,['US'],4.0,tt7829834,7.8,5581.0,8.217,7.6
1939,ts215037,Minecraft: Story Mode,SHOW,"MInecraft: Story Mode is an interactive, anima...",2018-01-01,TV-PG,52,NaN,['US'],1.0,tt10498322,5.6,347.0,NaN,NaN
2386,ts74805,A Little Help with Carol Burnett,SHOW,In this unscripted series starring comedy lege...,2018-01-01,TV-G,24,NaN,['US'],1.0,tt7204366,6.3,237.0,1.621,6.2
2658,ts265844,#ABtalks,SHOW,#ABtalks is a YouTube interview show hosted by...,2018-01-01,TV-PG,68,NaN,[],1.0,tt12635254,9.6,7.0,NaN,NaN
4274,tm1172010,The Lockdown Plan,MOVIE,NaN,2020-01-01,NaN,49,NaN,[],NaN,tt13079112,6.5,NaN,NaN,NaN
4648,tm1113921,In Vitro,MOVIE,'In Vitro' is an otherworldly rumination on me...,2019-01-01,NaN,27,NaN,[],NaN,tt10545994,7.7,NaN,NaN,NaN


In [17]:
c_titles=c_titles.dropna(subset=["genres"])
c_titles["genres"].isnull().sum()

np.int64(0)

##### c_credits中只有character有缺失值，而character角色一般不影响对演员的评分，所以将character的缺失值保留。

#### 2.2 处理重复数据  
根据数据变量的含义以及内容来看，c_titles里不应该存在每个变量值都相同的观察值，因此查看是否存在重复值。

In [18]:
c_titles.duplicated().sum()

np.int64(0)

In [19]:
c_credits.duplicated().sum()

np.int64(0)

#### 2.3 处理不一致数据  
从上可知c_titles以及c_credits输出值均为0，即不存在重复数据，接下来应当处理不一致数据。  
针对c_titles，不一致数据可能存在于genres\character\production_countries变量中，我们将查看是否存在多个不同值指代同一流派，以及多个不同值指代同一国家的情况。

In [20]:
c_titles["genres"].value_counts()

genres
drama            2827
comedy           2218
thriller         1180
action           1109
romance           955
crime             909
documentation     859
family            652
animation         630
fantasy           619
scifi             560
european          428
horror            366
history           254
music             244
reality           219
sport             170
war               155
western            39
Name: count, dtype: int64

##### 从统计数据上看，并不存在不一致称谓，但存在以空值代表的流派，在此可以将其删除。删除后再次进行查看。

In [21]:
c_titles=c_titles.query('genres!=""')
c_titles.query('genres==""')

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score


##### 清理时发现存在空列表，因此再加一个条件把空列表也给清除

In [22]:
c_titles=c_titles.query("production_countries!=''" and "production_countries!='[]'")
c_titles.query("production_countries==''")

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score


##### 接下来针对production_country列也是一样的操作：利用value_counts方法，得到production_countries的列表里面各个值的出现次数。

In [23]:
c_titles["production_countries"].value_counts()

production_countries
['US']          4587
['IN']          1545
['JP']           985
['KR']           618
['GB']           554
                ... 
['CN', 'GB']       1
['AF']             1
['BE', 'LU']       1
['TZ', 'GB']       1
['PH', 'FO']       1
Name: count, Length: 426, dtype: int64

##### 由于value_counts执行结果中有太多值，Pandas只会默认显示开头和结尾的一些值。要完整展示结果，可以把display.max_rows设置为None，即取消展示行数上限。  
但因为我们只是在当前调用value_counts时才需要看完整结果，所以可以结合option_context，只更改临时上限。

In [24]:
with pd.option_context('display.max_rows', None):
    print(c_titles['production_countries'].value_counts())

production_countries
['US']                                        4587
['IN']                                        1545
['JP']                                         985
['KR']                                         618
['GB']                                         554
['ES']                                         466
['FR']                                         348
['CA']                                         255
['MX']                                         198
['TR']                                         192
['BR']                                         186
['PH']                                         180
['DE']                                         179
['CN']                                         164
['AU']                                         150
['ID']                                         143
['IT']                                         139
['CA', 'US']                                   127
['GB', 'US']                                   123
['TW']    

##### 从以上输出结果来看，出品国家都用两位的国家代码来表示，除了里面存在一个的Lebanon值。  
Lebanon的国家代码是LB，出现了39次，说明此处数据不一致。LB和Lebanon都在表示同一国家，需要进行统一。  
把c_titles里，production_countries的"LB"和"Lebanon"统一为LB，并检查替换后是否还存在"LB"：

In [25]:
c_titles["production_countries"]=c_titles["production_countries"].replace({"Lebanon":"LB"})

##### 检查是否还有Lebanon存在，用explode将每一列的变量改为一个

In [26]:
with pd.option_context('display.max_rows', None):
    print(c_titles.explode('production_countries')['production_countries'].value_counts())

production_countries
['US']                                        4587
['IN']                                        1545
['JP']                                         985
['KR']                                         618
['GB']                                         554
['ES']                                         466
['FR']                                         348
['CA']                                         255
['MX']                                         198
['TR']                                         192
['BR']                                         186
['PH']                                         180
['DE']                                         179
['CN']                                         164
['AU']                                         150
['ID']                                         143
['IT']                                         139
['CA', 'US']                                   127
['GB', 'US']                                   123
['TW']    

##### 针对c_credits，不一致数据可能存在于role中，接下来需要查看是否存在多个不同值指代同一演职员类型的情况。

In [27]:
c_credits["role"].value_counts()

role
ACTOR       73251
DIRECTOR     4550
Name: count, dtype: int64

##### 据此可知role的情况应该只有这两种，因此不存在不一致数据，接下来要对role的数据类型进行转换成category，以此节省空间并明示role的类型有限。

In [28]:
c_credits["role"]=c_credits["role"].astype("category")
c_credits["role"]

0           ACTOR
1           ACTOR
2           ACTOR
3           ACTOR
4           ACTOR
           ...   
77796       ACTOR
77797       ACTOR
77798       ACTOR
77799       ACTOR
77800    DIRECTOR
Name: role, Length: 77801, dtype: category
Categories (2, str): ['ACTOR', 'DIRECTOR']

#### 2.4 处理无效或错误数据  
可以通过describe来对其进行快速了解。

In [29]:
o_titles.describe()

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
count,5850.000000,5850.000000,2106.000000,5368.000000,5.352000e+03,5759.000000,5539.000000
mean,2016.417094,76.888889,2.162868,6.510861,2.343938e+04,22.637925,6.829175
std,6.937726,39.002509,2.689041,1.163826,9.582047e+04,81.680263,1.170391
min,1945.000000,0.000000,1.000000,1.500000,5.000000e+00,0.009442,0.500000
25%,2016.000000,44.000000,1.000000,5.800000,5.167500e+02,2.728500,6.100000
50%,2018.000000,83.000000,1.000000,6.600000,2.233500e+03,6.821000,6.900000
75%,2020.000000,104.000000,2.000000,7.300000,9.494000e+03,16.590000,7.537500
max,2022.000000,240.000000,42.000000,9.600000,2.294231e+06,2274.044000,10.000000


##### 从以上统计信息来看，o_titles里不存在脱离现实意义的数值。  
o_credits由于不包含表示数值含义的变量，因此无需用describe检查。

### 3 整理数据

In [30]:
c_titles

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,crime,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972-01-01,R,109,drama,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972-01-01,R,109,action,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972-01-01,R,109,thriller,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5843,tm1097142,My Bride,MOVIE,The story follows a young man and woman who go...,2021-01-01,NaN,93,comedy,['EG'],NaN,tt14216488,5.0,327.0,2.545,5.300
5843,tm1097142,My Bride,MOVIE,The story follows a young man and woman who go...,2021-01-01,NaN,93,drama,['EG'],NaN,tt14216488,5.0,327.0,2.545,5.300
5845,tm1014599,Fine Wine,MOVIE,A beautiful love story that can happen between...,2021-01-01,NaN,100,romance,['NG'],NaN,tt13857480,6.8,45.0,1.466,NaN
5845,tm1014599,Fine Wine,MOVIE,A beautiful love story that can happen between...,2021-01-01,NaN,100,drama,['NG'],NaN,tt13857480,6.8,45.0,1.466,NaN


In [31]:
c_credits

,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR
...,...,...,...,...,...
77796,736339,tm1059008,Adelaida Buscato,María Paz,ACTOR
77797,399499,tm1059008,Luz Stella Luengas,Karen Bayona,ACTOR
77798,373198,tm1059008,Inés Prieto,Fanny,ACTOR
77799,378132,tm1059008,Isabel Gaona,Cacica,ACTOR


##### 两个df都有变量id，而且含义相同，因此可以用id作为键来链接两个DF；  
为了避免两表的数据互相空缺从而影响演员评分的统计，在此处用inner作为连接方法只保留两边都有的id;并将titles_add_credits赋值为合并后的结果

In [32]:
titles_add_credits=pd.merge(c_titles,c_credits,on="id",how="inner")
titles_add_credits.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,person_id,name,character,role
0,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,3748,Robert De Niro,Travis Bickle,ACTOR
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,14658,Jodie Foster,Iris Steensma,ACTOR
2,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,7064,Albert Brooks,Tom,ACTOR
3,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,3739,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179,48933,Cybill Shepherd,Betsy,ACTOR


##### 因为要统计的只是演员的评分，与导演无关，因此需要筛选出row=actor的观察值来，可以通过query来完成整个过程

In [34]:
actor_in_titles=titles_add_credits.query('role=="ACTOR"')

##### 为了挖掘高分演员，需要对流派和演员进行分组  
当需要找人时，相对于名字，person_id更加精准，也能够避免重名的情况  

In [36]:
g_genres_and_person_id=actor_in_titles.groupby(["genres","person_id"])

##### 分组后，我们只需要对imdb_score的值进行聚合计算，因此只提取imdb_score变量，随后调用mean，来计算各个流派影视作品中每位演员参演作品的平均imdb评分

In [38]:
imdb_mean=g_genres_and_person_id["imdb_score"].mean()
imdb_mean

genres   person_id
action   1000         6.866667
         100007       7.000000
         100013       6.400000
         100019       6.500000
         100020       6.500000
                        ...   
western  993735       6.500000
         998673       7.300000
         998674       7.300000
         998675       7.300000
         99940        4.000000
Name: imdb_score, Length: 168268, dtype: float64

##### 现在出现的是一个层次化的Series，需要调用reset_index来对层次化索引进行重置，得到更规整的DataFrame、

In [39]:
imdb_mean_re=imdb_mean.reset_index()
imdb_mean_re

,genres,person_id,imdb_score
0,action,1000,6.866667
1,action,100007,7.000000
2,action,100013,6.400000
3,action,100019,6.500000
4,action,100020,6.500000
...,...,...,...
168263,western,993735,6.500000
168264,western,998673,7.300000
168265,western,998674,7.300000
168266,western,998675,7.300000


##### 可以对上面的结果进行再次分组，从而能够便利我们更容易的找到各个流派里演员作品最高的平均得分是多少，以及对应的演员是谁。

In [42]:
genres_max=imdb_mean_re.groupby("genres")["imdb_score"].max()
genres_max

genres
action           9.3
animation        9.3
comedy           9.2
crime            9.5
documentation    9.1
drama            9.5
european         8.9
family           9.3
fantasy          9.3
history          9.1
horror           9.0
music            8.8
reality          8.9
romance          9.2
scifi            9.3
sport            9.1
thriller         9.5
war              8.8
western          8.9
Name: imdb_score, dtype: float64

##### 现在只有最高分，但却没有对应的人，此时可以与imdb_mean_re表连接。

In [49]:
pdmerge=pd.merge(imdb_mean_re,genres_max,on=["genres","imdb_score"],how="inner")
pdmerge

,genres,person_id,imdb_score
0,action,12790,9.3
1,action,1303,9.3
2,action,21033,9.3
3,action,336830,9.3
4,action,86591,9.3
...,...,...,...
131,war,826547,8.8
132,western,22311,8.9
133,western,28166,8.9
134,western,28180,8.9


##### 从上得知，最高分演员有时不止一位，可能会有多个演员平均得分相同。  
为得到ID对应的演员名字，可以和c_credits相连接，但是这个df中还有其他列变量，目前只需要person_id和name的对应，因此可以先将其提取出来,并删除重复变量。

In [46]:
df1_credits=c_credits[["person_id","name"]].drop_duplicates()
df1_credits.head()

,person_id,name
0,3748,Robert De Niro
1,14658,Jodie Foster
2,7064,Albert Brooks
3,3739,Harvey Keitel
4,48933,Cybill Shepherd


##### 此时就可以和前表df1_credits的最高分数据根据键person_id来进行合并，这样才能找到平均评分最高的演员。

In [51]:
df_merge=pd.merge(df1_credits,pdmerge,on="person_id")
df_merge

,person_id,name,genres,imdb_score
0,22311,Koichi Yamadera,western,8.9
1,1652,Lukas Haas,music,8.8
2,1641,Leonardo DiCaprio,music,8.8
3,28180,Unsho Ishizuka,western,8.9
4,28166,Megumi Hayashibara,western,8.9
...,...,...,...,...
131,439923,Steve Kerr,history,9.1
132,439923,Steve Kerr,sport,9.1
133,408553,Phil Jackson,documentation,9.1
134,408553,Phil Jackson,history,9.1


#### 此时的表难以一眼看出各个流派里平均评分最高的演员，此时可以通过DataFrame的sort_value方法排序，完成整理

In [56]:
A=df_merge.sort_values("genres").reset_index().drop("index",axis=1)
A

,person_id,name,genres,imdb_score
0,12790,Olivia Hack,action,9.3
1,86591,Cricket Leigh,action,9.3
2,336830,André Sogliuzzo,action,9.3
3,21033,Zach Tyler,action,9.3
4,1303,Jessie Flower,action,9.3
...,...,...,...,...
131,140181,Naoya Uchida,war,8.8
132,93017,Aoi Tada,western,8.9
133,28166,Megumi Hayashibara,western,8.9
134,28180,Unsho Ishizuka,western,8.9
